# Giám sát đám đông trên camera thật

Notebook áp dụng model đếm người đã train trên ShanghaiTech vào livestream camera đường phố, gồm đủ các bước của hệ thống. Camera demo dự kiến là **大阪 心斎橋 ユニクロ前 24時間ライブカメラ**; trong lúc chưa lấy được luồng của camera này, notebook tạm dùng **EarthCam Dublin** (khu phố đi bộ Temple Bar, đông người thật). Đổi camera chỉ cần đổi `YOUTUBE_URL` và `TZ`.

1. **Lấy frame** từ livestream YouTube (hoặc từ video / thư mục ảnh đã lưu).
2. **Đếm người** bằng density map trên từng frame.
3. **Vùng giám sát (ROI) và vùng loại trừ:** chỉ đếm người trong ROI, bỏ phần nằm trong vùng loại trừ (biển quảng cáo, màn hình, poster...).
4. **Khoanh vùng đám đông:** chia ROI thành ô lưới, đánh dấu ô đông bất thường.
5. **Phát hiện bất thường theo luật:** quá đông, ô quá dày, số người tăng đột ngột. Ngưỡng tính từ chính dữ liệu bình thường của camera.
6. **Kết quả:** video có lớp phủ, biểu đồ số người theo thời gian, nhật ký cảnh báo.

Model chỉ là một file checkpoint (`best_mc_*_partB.pth`), nên đổi sang model cải tiến chỉ cần đổi input. Nếu checkpoint có nhánh mask (C1), notebook vẽ thêm đường bao vùng có người.

**Chuẩn bị trên Kaggle:**
- *Settings → Internet: On* (để đọc livestream).
- *Add Input:* output của notebook baseline (có `best_mc_baseline_s42_partB.pth`).
- Dán link livestream vào `YOUTUBE_URL` ở phần 0.

In [ ]:
import glob, json, math, os, subprocess, sys, time
from datetime import datetime, timezone

import cv2
import matplotlib
import matplotlib.font_manager
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
print('PyTorch', torch.__version__, '| OpenCV', cv2.__version__, '| thiết bị:',
      torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')

## 0. Cấu hình

- **Nguồn hình** `SOURCE`: `'youtube'` đọc livestream trực tiếp; `'video'` đọc file video; `'frames'` đọc thư mục ảnh đã lưu từ lần chạy trước (nên dùng để các lần chạy sau so sánh được trên cùng dữ liệu).
- **ROI và vùng loại trừ:** mỗi vùng là một đa giác, tọa độ **chuẩn hóa 0–1** theo chiều rộng và chiều cao ảnh, nên không phụ thuộc độ phân giải. Phần 3 vẽ lưới tọa độ lên ảnh cam để chỉnh.
- **Ngưỡng cảnh báo:** để `None` thì tự tính từ đoạn đầu dữ liệu (`CALIB_FRACTION`), coi là lúc bình thường.

In [ ]:
# ---------- Nguồn hình ----------
SOURCE = 'youtube'            # 'youtube' | 'video' | 'frames'
# Camera. TẠM dùng EarthCam Dublin (khu Temple Bar, có người thật) trong khi chưa lấy được livestream Shinsaibashi.
# Đổi về Shinsaibashi: dán link đang phát của kênh FST.NET (大阪 心斎橋 ユニクロ前) và đặt TZ = 'Asia/Tokyo'.
YOUTUBE_URL = 'https://www.youtube.com/watch?v=3nyPER2kzqk'   # EarthCam Live: Dublin, Ireland
COOKIES_FILE = None           # nếu YouTube đòi xác minh "not a bot": file cookies.txt (định dạng Netscape) trong /kaggle/input
VIDEO_FILE = '/kaggle/input/**/recording/part_*.mkv'   # SOURCE = 'video': một file hoặc mẫu glob nhiều đoạn (bản ghi của record_shinsaibashi)
FRAMES_GLOB = None            # SOURCE = 'frames': ví dụ '/kaggle/input/shinsaibashi-frames/frames/*.jpg'
CAPTURE_MINUTES = 10          # livestream: số phút lấy frame (thời gian thực); video: chỉ lấy ngần này phút đầu, None = toàn bộ
CAPTURE_EVERY_S = 2.0         # lấy 1 frame mỗi ngần này giây
TZ = 'Europe/Dublin'          # giờ địa phương của camera (Shinsaibashi: 'Asia/Tokyo')

# ---------- Model ----------
CKPT = '/kaggle/input/notebooks/chinhde/baseline-mobilecount-tr-n-shanghaitech/best_mc_baseline_s42_partB.pth'
                              # None: tự tìm best_mc_*partB.pth trong /kaggle/input (ưu tiên baseline)
INFER_SCALE = 1.0             # phóng/thu frame trước khi đưa vào model (xem phần 2.2)

# ---------- Vùng giám sát ----------
# Đa giác, tọa độ (x, y) chuẩn hóa 0–1. Mặc định: cả khung trừ 20% phía trên. CHỈNH LẠI sau khi xem ảnh cam ở phần 3.
ROI = [[(0.0, 0.2), (1.0, 0.2), (1.0, 1.0), (0.0, 1.0)]]
EXCLUDE = []                  # ví dụ [[(0.75, 0.2), (1.0, 0.2), (1.0, 0.45), (0.75, 0.45)]]
CELL_PX = 64                  # cạnh ô lưới khoanh vùng đông, tính theo pixel frame gốc

# ---------- Luật cảnh báo ----------
CALIB_FRACTION = 0.3          # phần đầu dữ liệu dùng để học mức bình thường
K_SIGMA = 3.0                 # quá đông: số người > trung vị + K_SIGMA × độ lệch (ước lượng bền bằng MAD)
COUNT_LIMIT = None            # đặt số cụ thể để ghi đè ngưỡng tự tính
CELL_LIMIT = None             # ô quá dày: số người trong một ô; None = 1,3 × mức ô đông nhất thường gặp
MIN_CELL_LIMIT = 2.0
SURGE_WINDOW_S = 60           # tăng đột ngột: so với trung vị trong khoảng [t − 60 s, t − 20 s]
SURGE_ABS, SURGE_REL = 15, 0.5   # ... tăng hơn cả 15 người và 50%
SMOOTH = 3                    # làm mượt số đếm bằng trung vị trượt 3 frame
PERSIST = 3                   # luật phải đúng 3 frame liên tiếp mới cảnh báo (tránh báo nhầm do nhiễu)

# ---------- Đầu ra ----------
OUTPUT_FPS = 4
OUTPUT_WIDTH = 960

if 'DAN_LINK' in YOUTUBE_URL and SOURCE == 'youtube':
    print('CHƯA DÁN LINK LIVESTREAM vào YOUTUBE_URL.')

## 1. Model

Kiến trúc giống notebook baseline và notebook cải tiến (encoder MobileNetV2 + decoder Light-Weight RefineNet). Notebook đọc checkpoint và tự nhận biết có nhánh mask (C1) hoặc ASPP (C2) hay không.

In [ ]:
STRIDE, DENSITY_SCALE = 4, 100.0
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


class ChainedResidualPool(nn.Module):
    def __init__(self, channels, n_stages=4):
        super().__init__()
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(n_stages)])

    def forward(self, x):
        top = x
        for conv in self.convs:
            top = conv(self.pool(top))
            x = x + top
        return x


class ASPP(nn.Module):
    def __init__(self, cin, cout, rates=(1, 2, 4)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(cin, cout, 1 if r == 1 else 3, padding=0 if r == 1 else r, dilation=r, bias=False),
                          nn.ReLU(inplace=True)) for r in rates])
        self.image_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cin, cout, 1, bias=False), nn.ReLU(inplace=True))
        self.merge = nn.Sequential(nn.Conv2d(cout * (len(rates) + 1), cout, 1, bias=False), nn.ReLU(inplace=True))

    def forward(self, x):
        feats = [b(x) for b in self.branches]
        feats.append(F.interpolate(self.image_pool(x), size=x.shape[-2:], mode='bilinear', align_corners=False))
        return self.merge(torch.cat(feats, dim=1))


class CountModel(nn.Module):
    """MobileCount (MobileNetV2 + LW-RefineNet), tùy chọn nhánh mask C1 và ASPP C2."""

    def __init__(self, mask=False, aspp=False, channels=64):
        super().__init__()
        f = torchvision.models.mobilenet_v2(weights=None).features
        self.encoder = nn.ModuleList([f[:4], f[4:7], f[7:14], f[14:18]])
        self.project = nn.ModuleList([nn.Conv2d(c, channels, 1, bias=False) for c in (24, 32, 96)])
        self.deep = ASPP(320, channels) if aspp else nn.Sequential(nn.Conv2d(320, channels, 1, bias=False), nn.ReLU(inplace=True))
        self.crp = nn.ModuleList([ChainedResidualPool(channels) for _ in range(4)])
        self.fuse = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(3)])
        self.head = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(channels, 1, 1))
        self.mask_head = nn.Sequential(nn.Conv2d(channels, 32, 3, padding=1), nn.ReLU(inplace=True),
                                       nn.Conv2d(32, 1, 1)) if mask else None

    def forward(self, x):
        feats = []
        for stage in self.encoder:
            x = stage(x)
            feats.append(x)
        y = self.crp[3](self.deep(feats[3]))
        for i in (2, 1, 0):
            y = F.interpolate(self.fuse[i](y), size=feats[i].shape[-2:], mode='bilinear', align_corners=False)
            y = self.crp[i](F.relu(self.project[i](feats[i]) + y))
        den = self.head(y)
        if self.mask_head is None:
            return den, None
        logit = self.mask_head(y)
        return den * torch.sigmoid(logit), logit


def find_ckpt():
    if CKPT and os.path.exists(CKPT):
        return CKPT
    if CKPT:
        print(f'Không thấy {CKPT}, tự tìm checkpoint khác trong /kaggle/input.')
    found = glob.glob('/kaggle/input/**/best_mc_*partB.pth', recursive=True)
    if not found:
        raise FileNotFoundError('Không thấy best_mc_*partB.pth trong /kaggle/input: Add Input output của notebook baseline.')
    found.sort(key=lambda p: ('baseline' not in os.path.basename(p), p))
    print('Các checkpoint tìm thấy:', *found, sep='\n  ')
    return found[0]


ckpt_path = find_ckpt()
state = torch.load(ckpt_path, map_location='cpu')
HAS_MASK = any(k.startswith('mask_head') for k in state)
HAS_ASPP = any(k.startswith('deep.branches') for k in state)
model = CountModel(mask=HAS_MASK, aspp=HAS_ASPP)
model.load_state_dict(state)
model = model.to(device).eval()
print(f'Dùng {ckpt_path} | mask C1: {HAS_MASK} | ASPP C2: {HAS_ASPP} | '
      f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f} triệu tham số')


@torch.no_grad()
def predict_frame(rgb, scale=None):
    """rgb uint8 (H, W, 3) → density 1/4 (người/ô density) và mask xác suất (nếu có)."""
    scale = INFER_SCALE if scale is None else scale
    if scale != 1:
        rgb = cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    img = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
    h, w = img.shape[:2]
    H, W = math.ceil(h / 32) * 32, math.ceil(w / 32) * 32
    img = np.pad(img, ((0, H - h), (0, W - w), (0, 0)))
    x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))[None].to(device)
    with torch.autocast('cuda', dtype=torch.float16, enabled=device.type == 'cuda'):
        den, logit = model(x)
    hh, ww = math.ceil(h / STRIDE), math.ceil(w / STRIDE)
    den = den[0, 0].float().cpu().numpy()[:hh, :ww] / DENSITY_SCALE
    mask = None if logit is None else torch.sigmoid(logit.float())[0, 0].cpu().numpy()[:hh, :ww]
    return den, mask

## 2. Lấy frame

Livestream được đọc qua `yt-dlp` (lấy link stream HLS) và OpenCV. Mỗi `CAPTURE_EVERY_S` giây lưu một frame vào `/kaggle/working/frames` kèm `frames_index.csv` (thời điểm chụp).

**Nên lưu bộ frame này thành Kaggle Dataset** (*Output → New Dataset*). Lần sau đặt `SOURCE = 'frames'` để chạy lại trên đúng dữ liệu đó: so sánh model, chỉnh ROI hay ngưỡng mà không phụ thuộc livestream.

Nếu YouTube chặn máy chủ Kaggle (lỗi *Sign in to confirm you're not a bot*): xuất cookies YouTube từ trình duyệt ra `cookies.txt`, đưa vào input, đặt `COOKIES_FILE`. Hoặc quay màn hình livestream thành video, đưa vào input và dùng `SOURCE = 'video'`.

In [ ]:
FRAME_DIR = os.path.join(WORK_DIR, 'frames')


def youtube_stream_url(url, verbose=False):
    """Link luồng video (HLS) của livestream. Tự chọn luồng có hình, cao nhất nhưng không quá 1080p."""
    import yt_dlp
    opts = {'quiet': not verbose, 'no_warnings': not verbose, 'ignore_no_formats_error': True}
    if COOKIES_FILE:
        opts['cookiefile'] = COOKIES_FILE
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=False, process=False)
    if info.get('_type') == 'url' and info.get('url') and info['url'] != url:   # ví dụ link kênh /live → link video
        return youtube_stream_url(info['url'], verbose)
    fmts = [f for f in info.get('formats') or [] if f.get('url') and f.get('vcodec') not in (None, 'none')]
    if not fmts:
        if not verbose:   # chạy lại có in cảnh báo để thấy lý do YouTube không trả luồng
            return youtube_stream_url(url, verbose=True)
        raise RuntimeError(
            'YouTube không trả luồng video nào (xem cảnh báo của yt-dlp ở trên). Thường do: (1) thiếu deno để giải '
            'mã JavaScript của YouTube; (2) YouTube chặn IP của Kaggle, cần COOKIES_FILE; (3) link không phải '
            'livestream đang phát. Cách khác: quay màn hình livestream, dùng SOURCE = "video".')
    ok = [f for f in fmts if (f.get('height') or 0) <= 1080] or fmts
    best = max(ok, key=lambda f: ((f.get('height') or 0), 'm3u8' in (f.get('protocol') or '')))
    return best['url'], info.get('title'), best.get('height')


def capture_youtube():
    stream, title, height = youtube_stream_url(YOUTUBE_URL)
    print(f'Livestream: {title} ({height}p). Lấy frame trong {CAPTURE_MINUTES} phút, mỗi {CAPTURE_EVERY_S} s...')
    os.makedirs(FRAME_DIR, exist_ok=True)
    cap, rows, fails = cv2.VideoCapture(stream), [], 0
    t_end = time.time() + CAPTURE_MINUTES * 60
    next_t = time.time()
    while time.time() < t_end:
        if not cap.grab():
            fails += 1
            if fails > 5:
                print('Mất kết nối quá 5 lần, dừng lấy frame.')
                break
            cap.release(); time.sleep(3)
            cap = cv2.VideoCapture(youtube_stream_url(YOUTUBE_URL)[0])
            continue
        now = time.time()
        if now < next_t:
            continue
        ok, frame = cap.retrieve()
        if not ok:
            continue
        name = f'{int(now * 1000)}.jpg'
        cv2.imwrite(os.path.join(FRAME_DIR, name), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])
        rows.append({'name': name, 't': now})
        next_t = max(next_t + CAPTURE_EVERY_S, now)
        if len(rows) % 50 == 0:
            print(f'  {len(rows)} frame')
    cap.release()
    return rows


def recording_clock(files):
    """Thời điểm bắt đầu thật của từng đoạn video, nếu có recording_info.json của notebook record_shinsaibashi."""
    info_path = os.path.join(os.path.dirname(files[0]), 'recording_info.json')
    if not os.path.exists(info_path):
        return {}
    with open(info_path, encoding='utf-8') as fh:
        return {s['file']: s['start_epoch'] for s in json.load(fh).get('segments', [])}


def capture_video():
    os.makedirs(FRAME_DIR, exist_ok=True)
    files = sorted(glob.glob(VIDEO_FILE, recursive=True))
    if not files:
        raise FileNotFoundError(f'Không thấy video nào khớp {VIDEO_FILE}')
    clock = recording_clock(files)
    limit_s = CAPTURE_MINUTES * 60 if CAPTURE_MINUTES else float('inf')
    print(f'{len(files)} file video' + (', có giờ thật từ recording_info.json' if clock else ''))
    rows, offset = [], 0.0
    for fi, path in enumerate(files):
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        step, i = max(1, round(fps * CAPTURE_EVERY_S)), 0
        start = clock.get(os.path.basename(path))
        while offset + i / fps < limit_s:
            if not cap.grab():
                break
            if i % step == 0:
                ok, frame = cap.retrieve()
                if ok:
                    name = f'{fi:02d}_{i:08d}.jpg'
                    cv2.imwrite(os.path.join(FRAME_DIR, name), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])
                    rows.append({'name': name, 't': (start + i / fps) if start else offset + i / fps})
            i += 1
        cap.release()
        offset += i / fps
        print(f'  {os.path.basename(path)}: {i / fps / 60:.1f} phút, tổng {len(rows)} frame')
        if offset >= limit_s:
            break
    return rows


def load_frames():
    global FRAME_DIR
    paths = sorted(glob.glob(FRAMES_GLOB))
    FRAME_DIR = os.path.dirname(paths[0])
    index = os.path.join(FRAME_DIR, 'frames_index.csv')
    if os.path.exists(index):
        return pd.read_csv(index).to_dict('records')
    return [{'name': os.path.basename(p), 't': k * CAPTURE_EVERY_S} for k, p in enumerate(paths)]


t0 = time.time()
if SOURCE == 'youtube':
    # yt-dlp cần một trình chạy JavaScript (deno) để giải mã chống tải tự động của YouTube
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]', 'deno'], check=True)
    os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
    rows = capture_youtube()
elif SOURCE == 'video':
    rows = capture_video()
else:
    rows = load_frames()
if not rows:
    raise RuntimeError('Không lấy được frame nào.')

frames = pd.DataFrame(rows).sort_values('t').reset_index(drop=True)
frames['path'] = [os.path.join(FRAME_DIR, n) for n in frames['name']]
IS_CLOCK = frames['t'].iloc[0] > 1e9   # t là thời điểm thật (livestream) hay giây tính từ đầu video
if IS_CLOCK:
    frames['label'] = pd.to_datetime(frames['t'], unit='s', utc=True).dt.tz_convert(TZ).dt.strftime('%Y-%m-%d %H:%M:%S')
else:
    frames['label'] = [f'{int(s // 60):02d}:{s % 60:04.1f}' for s in frames['t']]
if SOURCE != 'frames':
    frames[['name', 't']].to_csv(os.path.join(FRAME_DIR, 'frames_index.csv'), index=False)

FRAME_H, FRAME_W = cv2.imread(frames['path'].iloc[0]).shape[:2]
print(f'{len(frames)} frame {FRAME_W}×{FRAME_H}, từ {frames.label.iloc[0]} đến {frames.label.iloc[-1]} '
      f'({time.time() - t0:.0f} s)')


def read_rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)


pick = np.linspace(0, len(frames) - 1, 4).astype(int)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, k in zip(axes, pick):
    ax.imshow(read_rgb(frames.path[k])); ax.set_title(frames.label[k]); ax.axis('off')
plt.tight_layout(); plt.show()

### 2.1. Model có dùng được trên camera này không?

Model học trên ShanghaiTech (ảnh đường phố Thượng Hải, 1024×768, góc chụp ngang tầm). Camera giám sát đường phố thường đặt cao, đầu người nhỏ hơn, có cảnh đêm và ánh đèn biển hiệu. Trước khi tin số đếm, cần xem density map có rơi đúng vào đầu người không, và có "thấy người" ở biển quảng cáo, màn hình hay không. Những chỗ đếm nhầm cố định như vậy sẽ được đưa vào vùng loại trừ ở phần 3.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for j, k in enumerate(pick):
    rgb = read_rgb(frames.path[k])
    den, mask = predict_frame(rgb)
    axes[0, j].imshow(rgb); axes[0, j].set_title(f'{frames.label[k]}'); axes[0, j].axis('off')
    axes[1, j].imshow(np.clip(den, 0, None), cmap='jet'); axes[1, j].axis('off')
    axes[1, j].set_title(f'density: {den.sum():.0f} người')
plt.suptitle('Frame camera và density map dự đoán (cả khung, chưa áp ROI)')
plt.tight_layout(); plt.show()

### 2.2. Độ nhạy theo kích thước đầu người

Model density nhạy với kích cỡ đầu người trong ảnh. Nếu đầu trên cam nhỏ hơn nhiều so với ShanghaiTech, phóng to frame (`INFER_SCALE > 1`) có thể giúp model nhận ra người tốt hơn. Bảng dưới đếm cùng frame ở nhiều tỉ lệ:

- Số đếm gần như không đổi theo tỉ lệ: model ổn định, giữ `INFER_SCALE = 1`.
- Số đếm thay đổi mạnh: model đang lệch miền. Cần **đếm tay vài frame** để biết tỉ lệ nào đúng, và về lâu dài nên gán nhãn frame cam để đo sai số và fine-tune.

In [ ]:
SCALES = [0.75, 1.0, 1.5, 2.0]
scale_rows = {}
for k in pick:
    rgb = read_rgb(frames.path[k])
    scale_rows[frames.label[k]] = {f'×{s}': round(float(predict_frame(rgb, s)[0].sum()), 1) for s in SCALES}
display(pd.DataFrame(scale_rows).T.rename_axis('frame'))

## 3. Vùng giám sát (ROI) và vùng loại trừ

- **ROI** (viền xanh): khu vực cần theo dõi, ví dụ lối đi bộ trước cửa hàng.
- **Vùng loại trừ** (tô xám, viền đỏ): nằm trong khung hình nhưng không theo dõi. Ví dụ biển quảng cáo, màn hình, poster có hình người (model hay đếm nhầm), hoặc khu vực không thuộc phạm vi giám sát.
- **Số người được giám sát** = tổng density trong ROI, trừ phần nằm trong vùng loại trừ.

Lưới tọa độ 0–1 trên ảnh giúp đọc tọa độ để sửa `ROI` và `EXCLUDE` ở phần 0, rồi chạy lại từ phần 3.

In [ ]:
def poly_px(poly, w, h):
    return np.round(np.array(poly, np.float32) * [w, h]).astype(np.int32)


_mask_cache = {}


def zone_masks(h, w):
    """Mask ROI và vùng loại trừ ở kích thước (h, w); 'watch' = trong ROI và ngoài vùng loại trừ."""
    if (h, w) not in _mask_cache:
        roi = np.zeros((h, w), np.uint8)
        for p in (ROI or [[(0, 0), (1, 0), (1, 1), (0, 1)]]):
            cv2.fillPoly(roi, [poly_px(p, w, h)], 1)
        exc = np.zeros((h, w), np.uint8)
        for p in EXCLUDE:
            cv2.fillPoly(exc, [poly_px(p, w, h)], 1)
        roi, exc = roi.astype(bool), exc.astype(bool)
        _mask_cache[(h, w)] = {'roi': roi, 'exclude': exc, 'watch': roi & ~exc}
    return _mask_cache[(h, w)]


rgb = read_rgb(frames.path[pick[1]])
den, _ = predict_frame(rgb)
m = zone_masks(FRAME_H, FRAME_W)
show = rgb.copy()
show[m['exclude']] = (show[m['exclude']] * 0.35).astype(np.uint8)
show[~m['roi']] = (show[~m['roi']] * 0.6).astype(np.uint8)
for p in ROI:
    cv2.polylines(show, [poly_px(p, FRAME_W, FRAME_H)], True, (0, 255, 0), 3)
for p in EXCLUDE:
    cv2.polylines(show, [poly_px(p, FRAME_W, FRAME_H)], True, (255, 0, 0), 3)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
axes[0].imshow(show, extent=(0, 1, 1, 0))
axes[0].set_xticks(np.arange(0, 1.01, 0.1)); axes[0].set_yticks(np.arange(0, 1.01, 0.1))
axes[0].grid(color='white', alpha=0.5, lw=0.8)
axes[0].set_title('ROI (xanh) và vùng loại trừ (đỏ), tọa độ chuẩn hóa')
dm = zone_masks(*den.shape)
axes[1].imshow(np.where(dm['watch'], np.clip(den, 0, None), np.nan), cmap='jet', extent=(0, 1, 1, 0))
axes[1].set_title(f"Density được tính: {den[dm['watch']].sum():.0f} người trong vùng giám sát "
                  f"(cả khung {den.sum():.0f}, trong vùng loại trừ {den[dm['exclude']].sum():.0f})")
plt.tight_layout(); plt.show()
if ROI == [[(0.0, 0.2), (1.0, 0.2), (1.0, 1.0), (0.0, 1.0)]] and not EXCLUDE:
    print('ROI đang là mặc định: hãy vẽ lại ROI và vùng loại trừ theo cảnh thật của camera.')

## 4. Đếm người và khoanh vùng đông trên toàn bộ frame

Với mỗi frame: số người cả khung, trong vùng giám sát, trong vùng loại trừ, và số người trong từng ô lưới `CELL_PX`×`CELL_PX` (chỉ tính phần thuộc vùng giám sát). Ô có số người vượt ngưỡng là nơi đám đông dày đặc.

*Hạn chế:* camera nhìn xiên, nên ô ở xa phủ diện tích mặt đất lớn hơn ô ở gần. Cùng số người, ô xa thực ra thưa hơn. Muốn tính đúng mật độ người/m² cần hiệu chỉnh phối cảnh (đo vài khoảng cách thật trên mặt đường).

In [ ]:
CELL_D = max(1, round(CELL_PX * INFER_SCALE / STRIDE))   # cạnh ô tính trên density map


def cell_sums(d, c=CELL_D):
    h, w = d.shape
    H, W = math.ceil(h / c) * c, math.ceil(w / c) * c
    return np.pad(d, ((0, H - h), (0, W - w))).reshape(H // c, c, W // c, c).sum(axis=(1, 3))


def analyze(rgb):
    den, mask = predict_frame(rgb)
    z = zone_masks(*den.shape)
    cells = cell_sums(np.where(z['watch'], den, 0))
    info = {'total': float(den.sum()), 'roi': float(den[z['watch']].sum()),
            'excluded': float(den[z['exclude']].sum()), 'max_cell': float(cells.max())}
    return info, den, mask, cells


records, dens, masks, cells_all = [], [], [], []
model_ms = []
t0 = time.time()
for k, path in enumerate(frames.path):
    rgb = read_rgb(path)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    ts = time.perf_counter()
    info, den, mask, cells = analyze(rgb)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    model_ms.append((time.perf_counter() - ts) * 1000)
    records.append(info); dens.append(den.astype(np.float16)); cells_all.append(cells.astype(np.float32))
    masks.append(None if mask is None else mask.astype(np.float16))
    if (k + 1) % 100 == 0:
        print(f'  {k + 1}/{len(frames)} frame')

counts = pd.concat([frames[['name', 't', 'label']], pd.DataFrame(records)], axis=1)
print(f'Xử lý {len(frames)} frame trong {time.time() - t0:.0f} s; model + đếm: '
      f'{np.median(model_ms[1:] or model_ms):.1f} ms/frame (≈ {1000 / np.median(model_ms[1:] or model_ms):.0f} FPS) trên '
      f'{torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"}, frame {FRAME_W}×{FRAME_H}')
display(counts[['total', 'roi', 'excluded', 'max_cell']].describe().loc[['mean', 'std', 'min', '50%', 'max']].round(1))

## 5. Luật phát hiện bất thường

Ba luật, đều giải thích được bằng lời:

| Luật | Ý nghĩa | Điều kiện |
|---|---|---|
| **Quá đông** | Số người trong vùng giám sát cao bất thường | số đếm (đã làm mượt) > trung vị + `K_SIGMA` × độ lệch của giai đoạn bình thường |
| **Ô quá dày** | Có chỗ người tụ dày đặc, dù tổng số người chưa cao | ô đông nhất > 1,3 × mức ô đông nhất thường gặp (phân vị 95 của giai đoạn bình thường) |
| **Tăng đột ngột** | Người dồn tới nhanh (tụ tập, sự cố) | số đếm hiện tại vượt trung vị của 20–60 giây trước hơn cả `SURGE_ABS` người và `SURGE_REL` |

- Ngưỡng tự học từ `CALIB_FRACTION` đầu dữ liệu, coi là lúc bình thường. Như vậy ngưỡng phù hợp với chính camera này thay vì một con số áp đặt.
- Độ lệch dùng MAD (độ lệch tuyệt đối trung vị), ít bị ảnh hưởng bởi vài frame đếm sai.
- Một luật phải đúng `PERSIST` frame liên tiếp mới thành cảnh báo, tránh báo nhầm do một frame nhiễu.

*Hướng mở rộng:* khi có dữ liệu nhiều ngày, học ngưỡng theo từng khung giờ (giờ cao điểm tối khác giờ sáng sớm).

In [ ]:
def calibrate(roi, max_cell):
    n = max(5, int(len(roi) * CALIB_FRACTION))
    r = pd.Series(roi[:n]).rolling(SMOOTH, min_periods=1).median()
    med = float(r.median())
    spread = 1.4826 * float((r - med).abs().median())
    count_limit = COUNT_LIMIT if COUNT_LIMIT is not None else med + K_SIGMA * max(spread, 0.05 * med, 1.0)
    cell_limit = CELL_LIMIT if CELL_LIMIT is not None else max(MIN_CELL_LIMIT, 1.3 * float(np.quantile(max_cell[:n], 0.95)))
    return {'n_calib': n, 'median': med, 'spread': spread, 'count_limit': count_limit, 'cell_limit': cell_limit}


def persist(flags, k):
    out, run = np.zeros(len(flags), bool), 0
    for i, v in enumerate(flags):
        run = run + 1 if v else 0
        out[i] = run >= k
    return out


def apply_rules(t, roi, max_cell, th):
    t, roi, max_cell = np.asarray(t, float), np.asarray(roi, float), np.asarray(max_cell, float)
    smooth = pd.Series(roi).rolling(SMOOTH, min_periods=1).median().to_numpy()
    cell_s = pd.Series(max_cell).rolling(SMOOTH, min_periods=1).median().to_numpy()
    ref = np.full(len(t), np.nan)
    for i in range(len(t)):
        past = (t >= t[i] - SURGE_WINDOW_S) & (t <= t[i] - SURGE_WINDOW_S / 3)
        if past.any():
            ref[i] = np.median(smooth[past])
    rise = smooth - ref
    surge = np.nan_to_num(rise, nan=0) > np.maximum(SURGE_ABS, SURGE_REL * np.nan_to_num(ref, nan=np.inf))
    out = pd.DataFrame({'smooth': smooth, 'ref': ref,
                        'quá đông': persist(smooth > th['count_limit'], PERSIST),
                        'ô quá dày': persist(cell_s > th['cell_limit'], PERSIST),
                        'tăng đột ngột': persist(surge, PERSIST)})
    out['cảnh báo'] = out[RULES].any(axis=1)
    return out


RULES = ['quá đông', 'ô quá dày', 'tăng đột ngột']


def alert_segments(t, labels, flags, values):
    """Gom các frame cảnh báo liên tiếp của từng luật thành sự kiện (bắt đầu, kết thúc, giá trị đỉnh)."""
    rows = []
    for rule in RULES:
        f = flags[rule].to_numpy()
        i = 0
        while i < len(f):
            if f[i]:
                j = i
                while j + 1 < len(f) and f[j + 1]:
                    j += 1
                rows.append({'luật': rule, 'bắt đầu': labels[i], 'kết thúc': labels[j],
                             'kéo dài (s)': round(t[j] - t[i] + CAPTURE_EVERY_S, 1),
                             'đỉnh số người': round(float(values[i:j + 1].max()), 1)})
                i = j + 1
            else:
                i += 1
    return pd.DataFrame(rows)

### 5.1. Kiểm tra luật trên chuỗi giả lập

Một đoạn livestream ngắn thường không có sự cố thật, nên trước hết kiểm tra các luật trên **chuỗi số đếm giả lập**: 10 phút bình thường (khoảng 60 người, dao động ngẫu nhiên), rồi người dồn tới lên khoảng 110 trong 30 giây, đồng thời một ô bị dồn dày. Các luật phải bắt được sự kiện này và không báo trong đoạn bình thường.

In [ ]:
rng = np.random.default_rng(0)
t_syn = np.arange(450) * 2.0
roi_syn = 60 + rng.normal(0, 4, len(t_syn))
cell_syn = 1.5 + rng.normal(0, 0.25, len(t_syn))
start = 300
roi_syn[start:start + 15] += np.linspace(0, 50, 15)
roi_syn[start + 15:] += 50
cell_syn[start + 10:] += 2.5

th_syn = calibrate(roi_syn, cell_syn)
res_syn = apply_rules(t_syn, roi_syn, cell_syn, th_syn)
fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(t_syn, roi_syn, lw=0.8, alpha=0.5, label='số đếm giả lập')
ax.plot(t_syn, res_syn['smooth'], label='làm mượt')
ax.axhline(th_syn['count_limit'], color='r', ls='--', label='ngưỡng quá đông')
ax.axvspan(0, t_syn[th_syn['n_calib'] - 1], color='gray', alpha=0.15, label='giai đoạn học mức bình thường')
for rule, c in zip(RULES, ['red', 'purple', 'orange']):
    on = res_syn[rule].to_numpy()
    ax.scatter(t_syn[on], np.full(on.sum(), roi_syn.max() + 5 + 4 * RULES.index(rule)), s=6, color=c, label=rule)
ax.axvline(t_syn[start], color='k', ls=':', label='bắt đầu sự kiện thật')
ax.set(xlabel='giây', ylabel='người', title='Kiểm tra luật trên dữ liệu giả lập')
ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()

seg_syn = alert_segments(t_syn, [f'{s:.0f}s' for s in t_syn], res_syn, roi_syn)
display(seg_syn)
false_alarm = res_syn['cảnh báo'].to_numpy()[:start].any()
first = np.flatnonzero(res_syn['cảnh báo'].to_numpy()[start:])
print('Báo nhầm trong đoạn bình thường:', 'CÓ' if false_alarm else 'không')
print('Phát hiện sự kiện sau:', f'{first[0] * 2.0:.0f} giây' if len(first) else 'KHÔNG phát hiện')

### 5.2. Áp dụng trên camera thật

In [ ]:
TH = calibrate(counts['roi'].to_numpy(), counts['max_cell'].to_numpy())
flags = apply_rules(counts['t'], counts['roi'], counts['max_cell'], TH)
counts = pd.concat([counts.drop(columns=[c for c in flags.columns if c in counts], errors='ignore'), flags], axis=1)
print(f"Mức bình thường (từ {TH['n_calib']} frame đầu): trung vị {TH['median']:.1f} người, độ lệch {TH['spread']:.1f}")
print(f"Ngưỡng quá đông: {TH['count_limit']:.1f} người | ngưỡng ô quá dày: {TH['cell_limit']:.2f} người/ô "
      f"({CELL_PX}×{CELL_PX} px) | tăng đột ngột: > {SURGE_ABS} người và > {SURGE_REL:.0%} trong {SURGE_WINDOW_S} s")

tx = (counts['t'] - counts['t'].iloc[0]) / 60
fig, axes = plt.subplots(2, 1, figsize=(18, 7), sharex=True)
axes[0].plot(tx, counts['total'], lw=0.8, alpha=0.5, label='cả khung')
axes[0].plot(tx, counts['roi'], lw=0.8, alpha=0.6, label='vùng giám sát')
axes[0].plot(tx, counts['smooth'], lw=1.6, label='vùng giám sát (làm mượt)')
axes[0].axhline(TH['count_limit'], color='r', ls='--', label='ngưỡng quá đông')
axes[0].axvspan(0, tx.iloc[TH['n_calib'] - 1], color='gray', alpha=0.15, label='học mức bình thường')
for rule, c in zip(RULES, ['red', 'purple', 'orange']):
    on = counts[rule].to_numpy()
    if on.any():
        axes[0].scatter(tx[on], counts['smooth'][on], s=10, color=c, zorder=3, label=rule)
axes[0].set(ylabel='người', title=f'Số người theo thời gian ({frames.label.iloc[0]} → {frames.label.iloc[-1]})')
axes[0].legend(ncol=4, fontsize=8)
axes[1].plot(tx, counts['max_cell'], lw=0.8, label='ô đông nhất')
axes[1].axhline(TH['cell_limit'], color='purple', ls='--', label='ngưỡng ô quá dày')
axes[1].set(xlabel='phút', ylabel='người/ô'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'counts_timeline.png'), dpi=110); plt.show()

alerts = alert_segments(counts['t'].to_numpy(), counts['label'].tolist(), counts, counts['smooth'].to_numpy())
counts.to_csv(os.path.join(WORK_DIR, 'counts_log.csv'), index=False)
alerts.to_csv(os.path.join(WORK_DIR, 'alerts_log.csv'), index=False)
print(f"{int(counts['cảnh báo'].sum())}/{len(counts)} frame có cảnh báo, {len(alerts)} sự kiện:")
display(alerts if len(alerts) else pd.DataFrame({'': ['Không có cảnh báo']}))

## 6. Video kết quả

Mỗi frame được vẽ: heatmap density, ROI (xanh), vùng loại trừ (xám, viền đỏ), ô quá dày (khung đỏ), đường bao vùng có người nếu model có nhánh mask C1 (vàng), và bảng thông tin (thời gian, số người, cảnh báo). Video lưu ở `/kaggle/working/monitor.mp4`.

In [ ]:
FONT_PATH = matplotlib.font_manager.findfont('DejaVu Sans')
HEAT_VMAX = float(np.quantile(np.stack([d.astype(np.float32) for d in dens[::max(1, len(dens) // 50)]]), 0.999)) + 1e-6


def render(k):
    rgb = read_rgb(frames.path[k])
    h, w = rgb.shape[:2]
    den = np.clip(dens[k].astype(np.float32), 0, None)
    norm = np.clip(cv2.resize(den, (w, h), interpolation=cv2.INTER_LINEAR) / HEAT_VMAX, 0, 1)
    heat = cv2.applyColorMap((norm * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
    a = (0.65 * norm)[..., None]
    img = (rgb * (1 - a) + heat * a).astype(np.uint8)
    z = zone_masks(h, w)
    img[z['exclude']] = (img[z['exclude']] * 0.35).astype(np.uint8)
    for p in EXCLUDE:
        cv2.polylines(img, [poly_px(p, w, h)], True, (255, 60, 60), 2)
    for p in ROI:
        cv2.polylines(img, [poly_px(p, w, h)], True, (0, 230, 0), 2)
    if masks[k] is not None:
        mk = (cv2.resize(masks[k].astype(np.float32), (w, h)) > 0.5).astype(np.uint8)
        cnts, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cnts, -1, (255, 230, 0), 2)
    cell_img = CELL_D * STRIDE / INFER_SCALE
    for r, c in zip(*np.nonzero(cells_all[k] > TH['cell_limit'])):
        x0, y0 = int(c * cell_img), int(r * cell_img)
        cv2.rectangle(img, (x0, y0), (int(x0 + cell_img), int(y0 + cell_img)), (255, 0, 0), 3)

    pil = Image.fromarray(img)
    draw = ImageDraw.Draw(pil, 'RGBA')
    fs = max(16, w // 60)
    font = ImageFont.truetype(FONT_PATH, fs)
    row = counts.iloc[k]
    lines = [(row['label'], (255, 255, 255)),
             (f"Vùng giám sát: {row['roi']:.0f} người (ngưỡng {TH['count_limit']:.0f})", (255, 255, 255)),
             (f"Cả khung: {row['total']:.0f} | ô đông nhất: {row['max_cell']:.1f}", (200, 200, 200))]
    lines += [(f'CẢNH BÁO: {rule.upper()}', (255, 70, 70)) for rule in RULES if row[rule]]
    pad = fs // 2
    draw.rectangle([0, 0, int(fs * 22), pad * 2 + len(lines) * int(fs * 1.3)], fill=(0, 0, 0, 150))
    for i, (text, color) in enumerate(lines):
        draw.text((pad, pad + i * int(fs * 1.3)), text, font=font, fill=color)
    if row['cảnh báo']:
        draw.rectangle([0, 0, w - 1, h - 1], outline=(255, 0, 0), width=8)
    return np.asarray(pil)


out_h = int(round(FRAME_H * OUTPUT_WIDTH / FRAME_W / 2) * 2)
video_path = os.path.join(WORK_DIR, 'monitor.mp4')
writer = None
for fourcc in ('avc1', 'mp4v'):
    writer = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*fourcc), OUTPUT_FPS, (OUTPUT_WIDTH, out_h))
    if writer.isOpened():
        break
render_ms = []
for k in range(len(frames)):
    ts = time.perf_counter()
    frame = render(k)
    render_ms.append((time.perf_counter() - ts) * 1000)
    writer.write(cv2.cvtColor(cv2.resize(frame, (OUTPUT_WIDTH, out_h), interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2BGR))
writer.release()
print(f'Video: {video_path} ({fourcc}, {len(frames)} frame, {OUTPUT_FPS} fps, '
      f'{os.path.getsize(video_path) / 1e6:.1f} MB). Vẽ lớp phủ: {np.median(render_ms):.0f} ms/frame.')

# Ảnh minh họa: ưu tiên frame có cảnh báo, còn lại rải đều
alert_idx = list(np.flatnonzero(counts['cảnh báo'].to_numpy()))
show_idx = sorted(set(alert_idx[::max(1, len(alert_idx) // 3)][:3]) | set(np.linspace(0, len(frames) - 1, 6 - min(3, len(alert_idx))).astype(int)))[:6]
fig, axes = plt.subplots(2, 3, figsize=(24, 10))
for ax, k in zip(axes.ravel(), show_idx):
    ax.imshow(render(k)); ax.axis('off')
for ax in axes.ravel()[len(show_idx):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'monitor_examples.jpg'), dpi=80); plt.show()

## 7. Tổng kết lần chạy

In [ ]:
total_ms = np.median(model_ms) + np.median(render_ms)
summary = pd.Series({
    'Checkpoint': os.path.basename(ckpt_path) + (' (mask C1)' if HAS_MASK else ''),
    'Nguồn': SOURCE, 'Số frame': len(frames), 'Khoảng thời gian': f'{frames.label.iloc[0]} → {frames.label.iloc[-1]}',
    'Frame': f'{FRAME_W}×{FRAME_H}, INFER_SCALE {INFER_SCALE}',
    'Người trong vùng giám sát (trung vị / lớn nhất)': f"{counts['roi'].median():.0f} / {counts['roi'].max():.0f}",
    'Người trong vùng loại trừ (trung bình)': f"{counts['excluded'].mean():.1f}",
    'Ngưỡng quá đông / ô quá dày': f"{TH['count_limit']:.0f} người / {TH['cell_limit']:.2f} người mỗi ô",
    'Số sự kiện cảnh báo': len(alerts),
    'Tốc độ model + đếm': f'{np.median(model_ms):.1f} ms/frame',
    'Tốc độ cả pipeline (kèm vẽ)': f'{total_ms:.0f} ms/frame ≈ {1000 / total_ms:.1f} FPS',
})
display(summary.to_frame('giá trị'))
print('\nFile trong', WORK_DIR)
for f in sorted(os.listdir(WORK_DIR)):
    p = os.path.join(WORK_DIR, f)
    if os.path.isfile(p):
        print(f'  {f:<28} {os.path.getsize(p) / 1024:10.1f} KB')
print(f'  frames/ ({len(frames)} ảnh + frames_index.csv): lưu thành Dataset để chạy lại với SOURCE = "frames"')